# Notebook 08L: Development learned peer and view feature preparation

**File:** `08l-development-learned-peer-view-features.ipynb`

This notebook reuses the sealed DINOv2-small outputs from Notebook 08G. It creates relative object features, fixed cross-view support measurements and automatic identity tracks for predeclared one, two, three and five-view schedules.

No evaluator labels, categories or source object identifiers are available to this notebook. The complete feature package is locked before the metric evaluation begins in Notebook 08M.

## Saved-output evidence copy

This notebook records the completed run for repository review. Its code and markdown match the clean runnable notebook. Private scene visuals and identifier-bearing output were removed. Safe aggregate tables and charts were retained.

Run stage: `08L Kaggle Version 3`  
Clean notebook SHA-256: `d64b31f66f67c6563fa40d942cd28c050bbdc05d61e270936ee7d485258910c9`  
Downloaded saved-output SHA-256: `13ebde3b802f85829e9fac444e1a3e2ce67144db71f9d75a157d59ac89f65382`


## Experiment boundary

- The established LocateAnything, SAM 2.1 and DINOv2-small pipeline is unchanged.
- The same 100-scene development screen is reused. No scene split is changed.
- The calibration and internal-lock roles remain unopened.
- The primary web range is two to five views. Single-view schedules measure the fallback case.
- Rotated schedules are sensitivity checks. They are not additional independent scenes.
- Every scene checkpoint is bound to the upstream prediction lock and feature protocol.

In [1]:
from pathlib import Path
from pathlib import PurePosixPath
import hashlib
import json
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
STAGE_NAME = "notebook-08l"
SOURCE_ARCHIVE_NAME = "learned-peer-view-source-v1.zip"
SOURCE_MANIFEST_NAME = "source-manifest.json"

def direct_matches(marker):
    matches = []
    for child in sorted(INPUT_ROOT.iterdir()):
        direct = child / marker
        if direct.exists():
            matches.append(direct.parent if direct.is_file() else direct)
        for nested in sorted(path for path in child.iterdir() if path.is_dir())[:20]:
            candidate = nested / marker
            if candidate.exists():
                matches.append(candidate.parent if candidate.is_file() else candidate)
    unique = sorted({path.resolve() for path in matches})
    if not unique:
        recursive = []
        for candidate in INPUT_ROOT.rglob(marker):
            recursive.append(candidate.parent if candidate.is_file() else candidate)
            if len(recursive) > 8:
                raise RuntimeError(f"Too many recursive matches for {marker}")
        unique = sorted({path.resolve() for path in recursive})
    return unique

def unpack_verified_source():
    manifests = sorted(INPUT_ROOT.rglob(SOURCE_MANIFEST_NAME))
    if len(manifests) != 1:
        raise RuntimeError(f"Expected one {SOURCE_MANIFEST_NAME} input, found {manifests}")
    manifest_path = manifests[0]
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    archive = manifest_path.parent / manifest.get("archive", "")
    if not archive.is_file():
        raise RuntimeError(f"Missing source archive beside {manifest_path}")
    observed_hash = sha256_file(archive)
    if observed_hash != manifest.get("archive_sha256"):
        raise RuntimeError("The source archive hash does not match its manifest")
    extraction_root = Path("/kaggle/temp") / f"vpi-learned-source-{observed_hash[:12]}"
    marker = extraction_root / "learned-peer-view-source/scripts/run_metric_feature_preparation.py"
    if marker.is_file():
        return extraction_root
    temporary = extraction_root.with_name(extraction_root.name + ".partial")
    shutil.rmtree(temporary, ignore_errors=True)
    temporary.mkdir(parents=True)
    with ZipFile(archive) as handle:
        members = handle.infolist()
        if not members or len(members) > 5000:
            raise RuntimeError("The source archive member count is outside the allowed range")
        if sum(member.file_size for member in members) > 100 * 1024 * 1024:
            raise RuntimeError("The source archive exceeds the extraction size limit")
        for member in members:
            relative = PurePosixPath(member.filename)
            if relative.is_absolute() or ".." in relative.parts:
                raise RuntimeError(f"Unsafe source archive member: {member.filename}")
            file_type = (member.external_attr >> 16) & 0o170000
            if file_type == 0o120000:
                raise RuntimeError(f"Source archive contains a symbolic link: {member.filename}")
        handle.extractall(temporary)
    if not (temporary / "learned-peer-view-source/scripts/run_metric_feature_preparation.py").is_file():
        raise RuntimeError("The verified source archive is missing the feature runner")
    shutil.rmtree(extraction_root, ignore_errors=True)
    temporary.replace(extraction_root)
    print(json.dumps({"event": "source_extracted", "archive_sha256": observed_hash}, sort_keys=True))
    return extraction_root

def find_bounded(marker):
    unique = direct_matches(marker)
    if not unique and marker == "scripts/run_metric_feature_preparation.py":
        unpack_verified_source()
        unique = direct_matches(marker)
    if len(unique) != 1:
        raise RuntimeError(f"Expected one {marker} input, found {unique}")
    return unique[0]

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

runtime_versions = {}
for package in ("numpy", "pandas", "scipy", "matplotlib", "pillow"):
    try:
        runtime_versions[package] = __import__("importlib.metadata").metadata.version(package)
    except Exception:
        runtime_versions[package] = "unavailable"

SOURCE_ROOT = find_bounded("scripts/run_metric_feature_preparation.py").parent
SCORING_ROOT = find_bounded("prediction_lock.json")
if not (SCORING_ROOT / "scenes").is_dir():
    raise RuntimeError("The scoring input is missing sealed scene predictions")

sys.path.insert(0, str(SOURCE_ROOT / "src"))
print(json.dumps({
    "event": "preflight_complete",
    "stage": STAGE_NAME,
    "source_root": str(SOURCE_ROOT),
    "scoring_root": str(SCORING_ROOT),
    "scoring_lock_sha256": sha256_file(SCORING_ROOT / "prediction_lock.json"),
    "runtime_versions": runtime_versions,
}, sort_keys=True))

{"event": "preflight_complete", "runtime_versions": {"matplotlib": "3.10.0", "numpy": "2.0.2", "pandas": "2.3.3", "pillow": "11.3.0", "scipy": "1.16.3"}, "scoring_lock_sha256": "12ce9bfc2bb401c29621d91000c53ff4fd39504fc28e55ff8a921e0b5443690d", "scoring_root": "/kaggle/input/notebooks/esreekarreddy/toysad-8k-development-scoring-comparison-full/scoring-comparison-08g", "source_root": "/kaggle/input/datasets/esreekarreddy/vpi-learned-peer-view-source/learned-peer-view-source-v1/learned-peer-view-source", "stage": "notebook-08l"}


## Prepare the sealed feature package

The command verifies the upstream lock before processing any scene. A valid result and receipt pair is reused on restart. Missing or mismatched pairs are recomputed.

In [2]:
OUTPUT_ROOT = WORKING_ROOT / "learned-peer-features-08l"
run_env = os.environ.copy()
run_env["PYTHONPATH"] = str(SOURCE_ROOT / "src")
command = [
    sys.executable,
    str(SOURCE_ROOT / "scripts/run_metric_feature_preparation.py"),
    "--scoring-root", str(SCORING_ROOT),
    "--output", str(OUTPUT_ROOT),
]
subprocess.run(command, check=True, timeout=1800, env=run_env)

{'event': 'feature_checkpoint', 'completed': 10, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 20, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 30, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 40, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 50, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 60, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 70, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 80, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 90, 'expected': 100, 'reused': 0}
{'event': 'feature_checkpoint', 'completed': 100, 'expected': 100, 'reused': 0}
{"binding_sha256": "e673866686549de6fc7fcf3c14185b1c5aee0c6f93405e701c506c72ca308ae5", "event": "feature_preparation_complete", "feature_count": 68, "feature_lock_sha256": "b39a1ca6b37ab238f74c969d5234a748cc6c

CompletedProcess(args=['/usr/bin/python3', '/kaggle/input/datasets/esreekarreddy/vpi-learned-peer-view-source/learned-peer-view-source-v1/learned-peer-view-source/scripts/run_metric_feature_preparation.py', '--scoring-root', '/kaggle/input/notebooks/esreekarreddy/toysad-8k-development-scoring-comparison-full/scoring-comparison-08g', '--output', '/kaggle/working/learned-peer-features-08l'], returncode=0)

## Verify the result

The feature lock is checked again in a new process. This catches incomplete writes or changed checkpoint bytes before packaging.

In [3]:
from visual_peer_inspector.workflows.learned_peer_features import feature_package_inventory

inventory = feature_package_inventory(OUTPUT_ROOT)
display(inventory)
display(__import__("pandas").read_csv(OUTPUT_ROOT / "shareable/feature_schema.csv"))

Counter({'locked_files': 204, 'scene_files': 100, 'feature_count': 68})

,feature_index,feature
0,0,rank_B0
1,1,robust_z_B0
2,2,available_B0
3,3,rank_B1_control
4,4,robust_z_B1_control
...,...,...
63,63,cross_similarity_fraction_ge_070
64,64,cross_anomaly_support_mean
65,65,cross_anomaly_support_max
66,66,view_count


## Create the downstream archive

Notebook 08M needs the locked feature package, including the scene checkpoints. This archive is a derived stage input, not a new dataset split.

In [4]:
import zipfile

downstream = WORKING_ROOT / "learned-peer-features-08l-downstream.zip"
temporary = downstream.with_suffix(".zip.partial")
with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_ROOT.parent))
    assert archive.testzip() is None
temporary.replace(downstream)
print(json.dumps({"archive": downstream.name, "bytes": downstream.stat().st_size, "sha256": sha256_file(downstream)}, sort_keys=True))

PACKAGE_ROOT = WORKING_ROOT / "learned-peer-packages-08l"
subprocess.run([
    sys.executable, str(SOURCE_ROOT / "scripts/package_metric_stage.py"),
    "--stage-root", str(OUTPUT_ROOT),
    "--output-root", str(PACKAGE_ROOT),
    "--prefix", "learned-peer-features-08l",
], check=True, timeout=300, env=run_env)
display(json.loads((PACKAGE_ROOT / "learned-peer-features-08l-package-receipt.json").read_text()))

{"archive": "learned-peer-features-08l-downstream.zip", "bytes": 9737831, "sha256": "35080f93349836d7201bb8eb1981715396704f4fe81129edafc172497ae019f1"}
{"event": "metric_stage_packaged", "repository": {"bytes": 2189, "file": "learned-peer-features-08l-repository-evidence.zip", "members": 2, "sha256": "9fb280869303f63edd617045c8f3f8d5874cbb1018f974717d14604be04ff5d4"}, "review": {"bytes": 2189, "file": "learned-peer-features-08l-review-evidence.zip", "members": 2, "sha256": "9fb280869303f63edd617045c8f3f8d5874cbb1018f974717d14604be04ff5d4"}, "schema_version": "metric-stage-package/v1"}


{'repository': {'bytes': 2189,
  'file': 'learned-peer-features-08l-repository-evidence.zip',
  'members': 2,
  'sha256': '9fb280869303f63edd617045c8f3f8d5874cbb1018f974717d14604be04ff5d4'},
 'review': {'bytes': 2189,
  'file': 'learned-peer-features-08l-review-evidence.zip',
  'members': 2,
  'sha256': '9fb280869303f63edd617045c8f3f8d5874cbb1018f974717d14604be04ff5d4'},
 'schema_version': 'metric-stage-package/v1'}

## Optional ClearML record

ClearML receives only the aggregate preparation receipt and feature schema. Tracking is isolated and time-limited, so a network or credential failure cannot invalidate the scientific output.

In [5]:
# Tracking runs only after the scientific outputs and checksums exist.
tracking_env = os.environ.copy()
tracking_status = "disabled"
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
        tracking_env[key] = secrets.get_secret(key)
    tracking_env.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
    tracking_env.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
    tracking_env.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
    tracking_site = WORKING_ROOT / "clearml-site"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--target", str(tracking_site), "clearml==2.1.10"],
        check=True,
        timeout=120,
    )
    tracking_env["PYTHONPATH"] = os.pathsep.join([str(tracking_site), str(SOURCE_ROOT / "src")])
    tracking_env.setdefault("CLEARML_API_DEFAULT_REQ_TIMEOUT", "10")
    subprocess.run(
        [sys.executable, str(SOURCE_ROOT / "scripts/log_learned_metric_clearml.py"),
         "--shareable-root", str(OUTPUT_ROOT / 'shareable'), "--receipt", str(OUTPUT_ROOT / 'tracking_receipt.json'),
         "--task-name", "08L learned peer-view feature preparation"],
        check=True,
        timeout=120,
        env=tracking_env,
    )
    tracking_status = json.loads(Path(OUTPUT_ROOT / 'tracking_receipt.json').read_text())["status"]
except Exception as error:
    receipt_path = Path(OUTPUT_ROOT / 'tracking_receipt.json')
    receipt_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_receipt = receipt_path.with_suffix(receipt_path.suffix + ".partial")
    temporary_receipt.write_text(json.dumps({"status": "failed", "error_type": type(error).__name__, "message": str(error)[:500]}, sort_keys=True))
    temporary_receipt.replace(receipt_path)
    tracking_status = "failed"
finally:
    shutil.rmtree(WORKING_ROOT / "clearml-site", ignore_errors=True)
print(json.dumps({"event": "tracking_finished", "status": tracking_status}, sort_keys=True))

{"event": "tracking_finished", "status": "failed"}


## Completion

The output is ready for Notebook 08M when the feature lock verifies and the downstream archive has a recorded SHA-256 value. The next notebook opens evaluator labels only after confirming those conditions.